In [ ]:
import os
import sys
sys.path.append('working_path')

import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
from scipy.stats import linregress
from statsmodels.stats.multitest import fdrcorrection
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

import joblib

from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import clone

from cca_zoo.linear import rCCA
from cca_zoo.model_selection import GridSearchCV, permutation_test_score
from cca_zoo.preprocessing import MultiViewPreprocessing

from confound_removal import ConfoundRemover

In [ ]:
df = pd.read_csv('working_path/cross_reversed.csv')

In [ ]:
"""
Define variables:
X set is sleep health
y set is depression
Age and sex are confounds 
"""
visit = '0.0'

# Define variables
Age = ['age']
Sex = ['Sex']
sleep = ['sleep_variables']
N_12 = ['n_12_depression_variables']
RDS_4 = ['rds4_depression_variables']
depression = N_12 + RDS_4

X_set = sleep
y_set = depression
confounds = Age # Sex not included for normalization 

# Extract numerical data for X_set, y_set, and confounds 
X_data = df[X_set].values
y_data = df[y_set].values
confounds_data = df[confounds].values
Sex_data = df[Sex].values

# Stack confounds with sex data
join_confounds = np.column_stack((confounds_data, Sex_data))

# Get indices for different parts of y_data
n_n12 = len(N_12)  # Number of binary variables
n_rds4 = len(RDS_4)  # Number of ordinal variables

In [ ]:
"""
The dataset will be splitted to 5 folds, each of them will be applied for rCCA and the other 4 for ML.
"""
# Define cross-validation strageties
# Define the CV for the entire dataset
n_splits_entire = 5
n_repeats_entire = 1

# Define the nested CV for ML
n_splits_outer_ML = 5
n_repeats_outer_ML = 1
n_splits_inner_ML = 5
n_repeats_inner_ML = 5

random_seed = 42

Outer_cv_entire = KFold(n_splits=n_splits_entire, shuffle=True, random_state=random_seed)

In [ ]:
# Initiating and fitting the rCCA model
n_latent_dimension = 1  # Only need the first latent dimension as the maximum canonical correlation
rcca = rCCA(latent_dimensions=n_latent_dimension)  # Initiating the model

In [ ]:
# Modified preprocessor for X data (sleep variables + confounds)
preprocessor_X = ColumnTransformer(
    transformers=[
        ('sleep', StandardScaler(), slice(0, X_data.shape[1])),  # Scale sleep variables
        ('confounds', StandardScaler(), slice(X_data.shape[1], X_data.shape[1] + confounds_data.shape[1])),  # Scale confounds
        ('sex', 'passthrough', [X_data.shape[1] + confounds_data.shape[1]])  # Sex variable
    ],
    sparse_threshold=0
)

# Modified preprocessor for y data (depression variables + confounds)
preprocessor_y = ColumnTransformer(
    transformers=[
        ('binary', 'passthrough', slice(0, n_n12)),  # No scaling for N12 binary variables
        ('ordinal', StandardScaler(), slice(n_n12, n_n12 + n_rds4)),  # Scale RDS4 ordinal variables
        ('confounds', StandardScaler(), slice(y_data.shape[1], y_data.shape[1] + confounds_data.shape[1])),  # Scale confounds
        ('sex', 'passthrough', [y_data.shape[1] + confounds_data.shape[1]])  # Sex variable
    ],
    sparse_threshold=0
)

In [ ]:
# Create a pipeline for analysis with confounds regressed
pipeline = Pipeline([
    ('preprocessor', MultiViewPreprocessing((preprocessor_X, preprocessor_y))),  # Apply preprocessing to both views
    ('deconfound', MultiViewPreprocessing((ConfoundRemover(n_confounds=join_confounds.shape[1]), 
                                         ConfoundRemover(n_confounds=join_confounds.shape[1])))),
    ('rcca', rcca),
])

In [ ]:
"""
The following are for hyperparameter tuning
"""
# Custom scoring function
def scorer(estimator, views):
    dim_corrs = estimator.score(views)  
    return dim_corrs.mean()

# Define grid of potential regularization parameters in linear approach
def search_grid(linear_X, linear_y):
    param_grid = {'rcca__c': [linear_X, linear_y]}
    return param_grid

linear_X = np.logspace(-4, 0, 10)
linear_y = np.logspace(-4, 0, 10)
param_grid = search_grid(linear_X, linear_y)

In [ ]:
def cross_validation(X_data, y_data, join_confounds, CCA_idx, ML_idx):
    """
    Prepare training and validation data by stacking X_data, y_data with confounds.
    """
    X_CCA = np.hstack((X_data[CCA_idx], join_confounds[CCA_idx]))
    y_CCA = np.hstack((y_data[CCA_idx], join_confounds[CCA_idx]))
    X_ML = np.hstack((X_data[ML_idx], join_confounds[ML_idx]))
    y_ML = np.hstack((y_data[ML_idx], join_confounds[ML_idx]))
    return X_CCA, y_CCA, X_ML, y_ML

In [ ]:
def train_model(pipeline, param_grid, X_CCA, y_CCA, cv, scorer):
    """
    hyperparameter tuning with GridSearchCV.
    """
    ridge = GridSearchCV(pipeline, param_grid=param_grid, cv=cv, n_jobs=-1, scoring=scorer)
    ridge.fit((X_CCA, y_CCA))
    return ridge.best_estimator_

In [ ]:
def check_confounds(model, X_CCA, y_CCA, join_confounds, train_idx):
    """
    Check if the confounds are regressed out by inspecting the correlation 
    between deconfounded data and confounds.
    """

    # Get deconfounded data
    deconfounded_data = model.named_steps['deconfound'].transform([X_CCA, y_CCA])
    
    X_deconfounded = deconfounded_data[0]
    y_deconfounded = deconfounded_data[1]
    join_confounds_train = join_confounds[train_idx]

    # Calculate mean correlation across all features
    confound_correlation_X = np.mean([
        np.corrcoef(X_deconfounded[:, i].ravel(), join_confounds_train[:, k].ravel())[0, 1]
        for i in range(X_deconfounded.shape[1])
        for k in range(join_confounds.shape[1])
    ])
    
    confound_correlation_Y = np.mean([
        np.corrcoef(y_deconfounded[:, i].ravel(), join_confounds_train[:, k].ravel())[0, 1]
        for i in range(y_deconfounded.shape[1])
        for k in range(join_confounds.shape[1])
    ])
    
    return confound_correlation_X, confound_correlation_Y

In [ ]:
def canonical_correlations_CCA(model, X_CCA, y_CCA):
    """
    Calculate canonical correlations for each dimension in the latent space.
    """
    # Apply preprocessing and deconfounding to the training data 
    CCA_preprocessed = model.named_steps['preprocessor'].transform([X_CCA, y_CCA])
    CCA_deconfounded = model.named_steps['deconfound'].transform([CCA_preprocessed[0], CCA_preprocessed[1]])

    X_CCA_deconf = CCA_deconfounded[0]
    y_CCA_deconf = CCA_deconfounded[1]

    # Calculate canonical correlation coefficient for training set
    CCA_canonical_corrs = model.named_steps['rcca'].pairwise_correlations([X_CCA_deconf, y_CCA_deconf])[0, 1]

    return CCA_canonical_corrs

In [ ]:
def collect_canonical_loadings_CCA(model, n_latent_dimension, X_CCA, y_CCA):
    """
    Calculate canonical loadings (correlations between original variables and their canonical variates)
    """
    loadingsX_CCA_list = []
    loadingsY_CCA_list = []

    # Apply preprocessing and deconfounding to the test data 
    CCA_preprocessed = model.named_steps['preprocessor'].transform([X_CCA, y_CCA])
    CCA_deconfound = model.named_steps['deconfound'].transform([CCA_preprocessed[0], CCA_preprocessed[1]])

    X_CCA_deconf = CCA_deconfound[0]
    y_CCA_deconf = CCA_deconfound[1]

    for comp in range(n_latent_dimension):
        # Get canonical variates 
        all_loadings = model.named_steps['rcca'].canonical_loadings_([X_CCA_deconf, y_CCA_deconf])

        X_loadings_CCA = all_loadings[0]
        y_loadings_CCA = all_loadings[1]

        # Append loadings for the current component
        loadingsX_CCA_list.append(X_loadings_CCA[:, comp])
        loadingsY_CCA_list.append(y_loadings_CCA[:, comp])
    return loadingsX_CCA_list, loadingsY_CCA_list

In [ ]:
def collect_weights_CCA(model, n_latent_dimension):
    """
    Calculate loadings (weights for each view)
    """
    weightsY_list = []

    for comp in range(n_latent_dimension):
        # Get canonical variates 
        all_weights = model.named_steps['rcca'].loadings_

        y_weights = all_weights[1]

        # Append loadings for the current component
        weightsY_list.append(y_weights[:, comp])
    return weightsY_list

In [ ]:
def flip_loadings_weights(X_loadings_list, y_loadings_list, weightsY_list):
    """
    Ensures all fold loadings are in the same direction as the first fold.
    
    Parameters:
    X_loadings_list: List of X loadings from all folds
    y_loadings_list: List of y loadings from all folds
    weightsY_list: List of Y weights from all folds
    """
    # Initialize accumulated lists with first fold's loadings
    X_loadings_accum = []
    y_loadings_accum = []
    flipped_weights_accum = []
    corr_accum = []

    # Process subsequent folds
    for i in range(len(X_loadings_list)):
        # Calculate correlation with first fold
        corr = np.corrcoef(X_loadings_list[0], 
                        X_loadings_list[i])[0, 1]
        corr_accum.append(corr)

        # Flip if negative correlation
        if corr < 0:
            flipped_X = -np.array(X_loadings_list[i])
            flipped_y = -np.array(y_loadings_list[i])
            flipped_y_weight = -np.array(weightsY_list[i])
        else:
            flipped_X = X_loadings_list[i]
            flipped_y = y_loadings_list[i]
            flipped_y_weight = weightsY_list[i]
            
        X_loadings_accum.append(flipped_X)
        y_loadings_accum.append(flipped_y)
        flipped_weights_accum.append(flipped_y_weight)
    
    return corr, flipped_X, flipped_y, X_loadings_accum, y_loadings_accum, corr_accum, flipped_weights_accum

In [ ]:
def flip_loadings_ML(X_loadings_ML_list, y_loadings_ML_list):
    """
    Ensures all fold loadings are in the same direction as the first fold.
    Handles accumulation internally.
    
    Parameters:
    X_loadings_ML_list: List of X loadings from all folds in ML data
    y_loadings_ML_list: List of y loadings from all folds in ML data
    """
    # Initialize accumulated lists with first fold's loadings
    X_loadings_ML_accum = []
    y_loadings_ML_accum = []

    # Process subsequent folds
    for i in range(len(X_loadings_ML_list)):
        # Calculate correlation with first fold
        corr_ML = np.corrcoef(X_loadings_ML_list[0], 
                        X_loadings_ML_list[i])[0, 1]

        # Flip if negative correlation
        if corr_ML < 0:
            flipped_X_ML = -np.array(X_loadings_ML_list[i])
            flipped_y_ML = -np.array(y_loadings_ML_list[i])
        else:
            flipped_X_ML = X_loadings_ML_list[i]
            flipped_y_ML = y_loadings_ML_list[i]
            
        X_loadings_ML_accum.append(flipped_X_ML)
        y_loadings_ML_accum.append(flipped_y_ML)
    
    return corr_ML, flipped_X_ML, flipped_y_ML, X_loadings_ML_accum, y_loadings_ML_accum

In [ ]:
def flip_latent_var_CCA(X_latent_var_list, y_latent_var_list):
    """
    Ensures all fold latent variates are in the same direction as the first fold.
    """
    # Initialize accumulated lists with first fold's loadings
    X_latent_var_accum = []
    y_latent_var_accum = []
    corr_latent_CCA_accum = []

    # Process subsequent folds
    for i in range(len(X_latent_var_list)):
        # Calculate correlation with first fold
        corr_latent_var_CCA = np.corrcoef(X_latent_var_list[0].ravel(), 
                        X_latent_var_list[i].ravel())[0, 1]
        corr_latent_CCA_accum.append(corr_latent_var_CCA)

        # Flip if negative correlation
        if corr_latent_var_CCA < 0:
            flipped_latent_X_CCA = -np.array(X_latent_var_list[i])
            flipped_latent_y_CCA = -np.array(y_latent_var_list[i])
        else:
            flipped_latent_X_CCA = X_latent_var_list[i]
            flipped_latent_y_CCA = y_latent_var_list[i]
            
        X_latent_var_accum.append(flipped_latent_X_CCA)
        y_latent_var_accum.append(flipped_latent_y_CCA)
    
    return corr_latent_var_CCA, flipped_latent_X_CCA, flipped_latent_y_CCA, X_latent_var_accum, y_latent_var_accum, corr_latent_CCA_accum

In [ ]:
def flip_latent_var_ML(X_latent_var_ML_list, y_latent_var_ML_list):
    """
    Ensures all fold latent variates are in the same direction as the first fold.
    
    Parameters:
    X_loadings_list: List of X loadings from all folds
    y_loadings_list: List of y loadings from all folds
    weightsY_list: List of Y weights from all folds
    """
    # Initialize accumulated lists with first fold's loadings
    X_latent_var_ML_accum = []
    y_latent_var_ML_accum = []
    corr_latent_ML_accum = []

    # Process subsequent folds
    for i in range(len(X_latent_var_ML_list)):
        # Calculate correlation with first fold
        corr_latent_var_ML = np.corrcoef(X_latent_var_ML_list[0].ravel(), 
                        X_latent_var_ML_list[i].ravel())[0, 1]
        corr_latent_ML_accum.append(corr_latent_var_ML)

        # Flip if negative correlation
        if corr_latent_var_ML < 0:
            flipped_latent_X_ML = -np.array(X_latent_var_ML_list[i])
            flipped_latent_y_ML = -np.array(y_latent_var_ML_list[i])
        else:
            flipped_latent_X_ML = X_latent_var_ML_list[i]
            flipped_latent_y_ML = y_latent_var_ML_list[i]
            
        X_latent_var_ML_accum.append(flipped_latent_X_ML)
        y_latent_var_ML_accum.append(flipped_latent_y_ML)
    
    return corr_latent_var_ML, flipped_latent_X_ML, flipped_latent_y_ML, X_latent_var_ML_accum, y_latent_var_ML_accum, corr_latent_ML_accum

In [ ]:
def save_split_datasets(df, ML_idx, CCA_idx, transformed_y_ML_depr, original_column_names, i):
    """
    Save split datasets to separate CSV files with additional columns and maintain eid mapping
    
    Parameters:
    df: Original dataframe
    ML_idx: Indices for ML dataset
    CCA_idx: Indices for CCA dataset
    transformed_y_ML_depr: Transformed depression scores
    original_column_names: List of original column names for transformed values
    i: Current fold number
    cca_data_accum: Accumulated CCA data
    ml_data_accum: Accumulated ML data
    """
    # Extract participants ID that matches ML_idx and CCA_idx
    ML_eid = df.iloc[ML_idx]['eid'].tolist()
    CCA_eid = df.iloc[CCA_idx]['eid'].tolist()
    
    # Copy a dataframe for ML and CCA modification
    df_for_ML = df.copy()
    df_for_CCA = df.copy()

    # Extract ML and CCA data corresponding to the specific eid
    ML_data = df_for_ML[df_for_ML['eid'].isin(ML_eid)].copy()
    CCA_data = df_for_CCA[df_for_CCA['eid'].isin(CCA_eid)].copy()
    
    # Add ML_idx and ML_eid as columns for reference
    ML_data[f'ML_idx_in_fold_{i+1}'] = ML_idx
    ML_data[f'ML_eid_in_fold_{i+1}'] = ML_eid
    
    # Add transformed depression columns to ML data
    for j, col_name in enumerate(original_column_names):
        ML_data[f'Trans_{col_name}_in_fold_{i+1}'] = transformed_y_ML_depr[:, j]
    
    # Add CCA_idx as a column for reference
    CCA_data[f'CCA_idx_in_fold_{i+1}'] = CCA_idx
    CCA_data[f'CCA_eid_in_fold_{i+1}'] = CCA_eid

    # Save ML dataset
    ML_data.to_csv(f'working_path/ML_cross_fold_{i+1}.csv', index=False)
    
    # Save CCA dataset
    CCA_data.to_csv(f'working_path/CCA_cross_fold_{i+1}.csv', index=False)

In [ ]:
def save_info_for_each_fold(i, CCA_idx, X_latent_var, y_latent_var, all_CCA_latent_df):
    """
    Save the latent variables and loadings for each view for later visualization for each fold
    """
    
    # Reduce dimensionality 
    X_latent_var = np.array(X_latent_var).flatten()
    y_latent_var = np.array(y_latent_var).flatten()

    # Get the eid for the corresponding fold
    CCA_eid = df.iloc[CCA_idx]['eid'].tolist()

    fold_df = pd.DataFrame({
        'eid': CCA_eid,
        'fold': i + 1,
        'X_latent_var': X_latent_var,
        'y_latent_var': y_latent_var
    })

    # Append to accumulator
    all_CCA_latent_df = pd.concat([all_CCA_latent_df, fold_df], ignore_index=True)
    return all_CCA_latent_df

In [ ]:
def save_loadings_for_each_fold(i, X_loadings, y_loadings, all_loadings_df):
    """
    Save canonical loadings for each fold
    """
    X_loadings = np.array(X_loadings).flatten()
    y_loadings = np.array(y_loadings).flatten()

    fold_df_X = pd.DataFrame({
        'fold': i + 1,
        'view': 'Sleep',
        'loading': X_loadings
    })

    fold_df_y = pd.DataFrame({
        'fold': i + 1,
        'view': 'Depression',
        'loading': y_loadings
    })

    fold_df = pd.concat([fold_df_X, fold_df_y], ignore_index=True)
    all_loadings_df = pd.concat([all_loadings_df, fold_df], ignore_index=True)
    return all_loadings_df

In [ ]:
def latent_var (model, X_set, y_set):
    """
    Calculate the latent variables for each view
    """
    CCA_preprocessed = model.named_steps['preprocessor'].transform([X_set, y_set])
    CCA_denoised = model.named_steps['deconfound'].transform([CCA_preprocessed[0], CCA_preprocessed[1]])
    Latnet_var = model.named_steps['rcca'].transform([CCA_denoised[0], CCA_denoised[1]])

    X_latent_var = Latnet_var[0]
    y_latent_var = Latnet_var[1]

    return X_latent_var, y_latent_var

In [ ]:
def explained_var (model, X_CCA, y_CCA):
    """
    Calculate the explained variance of the each view in each view
    """
    # Apply preprocessing and deconfounding to the training data 
    CCA_preprocessed = model.named_steps['preprocessor'].transform([X_CCA, y_CCA])
    CCA_deconfounded = model.named_steps['deconfound'].transform([CCA_preprocessed[0], CCA_preprocessed[1]])

    X_CCA_deconf = CCA_deconfounded[0]
    y_CCA_deconf = CCA_deconfounded[1]

    # Calculate canonical correlation coefficient for training set
    exp_var = model.named_steps['rcca'].explained_variance([X_CCA_deconf, y_CCA_deconf])

    exp_var_X1 = exp_var[0][0]  # X-view, 1st component
    exp_var_Y1 = exp_var[1][0]  # Y-view, 1st component

    return exp_var_X1, exp_var_Y1

In [ ]:
# Perform permutation testing for in training set
def permutation_test_rcca (best_model, X_CCA, y_CCA, X_ML, y_ML, n_perm = 1000):
    perm_corrs = np.zeros(n_perm)

    for i in range(n_perm):
        # Shuffle the training and test data
        perm_train_idx = np.random.permutation(len(y_CCA))
        y_train_perm = y_CCA[perm_train_idx, :]

        perm_test_idx = np.random.permutation(len(y_ML))
        y_test_perm = y_ML[perm_test_idx, :]

        # Clone the model with the same tuned hyperparameters
        model_perm = clone(best_model)

        # Fit on the shuffled data
        model_perm.fit([X_CCA, y_train_perm])

        # Transform on the shuffled ML data for model evaluation
        model_perm.transform([X_ML, y_test_perm])

        # Calculate the canonical correlation coefficient of the permuted model on the ML data
        perm_corrs[i] = canonical_correlations_CCA(model_perm, X_ML, y_test_perm).item()
    
    return perm_corrs

In [ ]:
"""
To get the best hyper-parameter for each outer fold and refit it on the entire CCA data and get the weight for latent depression
"""
# Initialize lists to store results
best_hyperparameters_fold = []
weightsY_list = []

CCA_correlations = []
CCA_correlations_ML = []
loadingsX_CCA_list = []
loadingsY_CCA_list = []
loadingsX_ML_list = []
loadingsY_ML_list = []
weightsY_list = []

all_CCA_latent_df = pd.DataFrame()
all_CCA_latent_ML_df = pd.DataFrame()
all_loadings_df = pd.DataFrame()
all_loadings_ML_df = pd.DataFrame()
weight_y_df = pd.DataFrame()

X_latent_var_list = []
y_latent_var_list = []

X_latent_var_ML_list = []
y_latent_var_ML_list = []

exp_X_list = []
exp_y_list = []

permuted_pvals = []
explained_variance_sleep_list = []
explained_variance_depression_list = []
redundancy_index_list = []

# Get the best hyper-parameter for each outer fold
for i, (ML_idx, CCA_idx) in enumerate(Outer_cv_entire.split(df)):
    # Prepare data in CV
    X_ML, y_ML, X_CCA, y_CCA = cross_validation(X_data, y_data, join_confounds, ML_idx,CCA_idx)

    # Train model
    best_model = train_model(pipeline, param_grid, X_CCA, y_CCA, Outer_cv_entire, scorer)

    # Get the best hyperparameters for this fold
    best_hyperparameters = best_model.named_steps['rcca'].get_params()
    best_hyperparameters_fold.append(best_hyperparameters)

    # Check confounds
    confound_correlation_X, confound_correlation_Y = check_confounds(best_model, X_CCA, y_CCA, join_confounds, CCA_idx)

    # Print the best hyperparameter 
    print(f'Information for outer fold {i+1} in CCA data:')
    print(f'Best hyperparameter: {best_hyperparameters}')
    print('\n')
    print(f"Correlation between deconfounded X and confounds: {confound_correlation_X}")
    print(f"Correlation between deconfounded Y and confounds: {confound_correlation_Y}")
     
    # Fit the best hyperparameters to the entire CCA data
    best_model.fit([X_CCA, y_CCA])

    # Calculate the weights for the entire CCA data with the best hyperparameters for each fold
    y_weights = collect_weights_CCA(best_model, n_latent_dimension)
    weightsY_list.append(y_weights)

    print (f"Weight for depression in fold {i+1}: {y_weights}\n")

    # Add weight to the dataframe
    weight_y_df[f'weight_depr_cross_fold_{i+1}'] = y_weights

    # Calculate canonical correlations for the entire CCA data for each fold
    CCA_canonical_corrs = canonical_correlations_CCA(best_model, X_CCA, y_CCA)
    CCA_correlations.append(CCA_canonical_corrs)

    # Collect the explained variance for each view in the first latent dimension
    exp_var_X1, exp_var_Y1 = explained_var (best_model, X_CCA, y_CCA)
    exp_X_list.append(exp_var_X1)
    exp_y_list.append(exp_var_Y1)

    # Calculate the latent variables for sleep and depression
    X_latent_var, y_latent_var = latent_var(best_model, X_CCA, y_CCA)

    X_latent_var_list.append(X_latent_var)
    y_latent_var_list.append(y_latent_var)

    # Get loadings for each dimension for the entire CCA data
    X_loadings_CCA, y_loadings_CCA = collect_canonical_loadings_CCA(best_model, n_latent_dimension, X_CCA, y_CCA)
    
    loadingsX_CCA_list.append(X_loadings_CCA)
    loadingsY_CCA_list.append(y_loadings_CCA)

    # Apply preprocessing and deconfounding to the ML data
    ML_preprocessed = best_model.named_steps['preprocessor'].transform([X_ML, y_ML])
    ML_deconfounded = best_model.named_steps['deconfound'].transform([ML_preprocessed[0], ML_preprocessed[1]])

    # Extract the preprocessed depression data for ML analysis
    y_data_deconf = ML_deconfounded[1]

    # Flip the loadings to ensure each fold is in the same direction with the 1st one
    corr, flipped_X, flipped_y, X_loadings_accum, y_loadings_accum, corr_accum, flipped_weights_accum = flip_loadings_weights(
        X_loadings_list=loadingsX_CCA_list, 
        y_loadings_list=loadingsY_CCA_list,
        weightsY_list = weightsY_list
    )

    # Flip the latent variates for CCA fold to ensure each fold is in the same direction with the 1st one
    corr_latent_var_CCA, flipped_latent_X_CCA, flipped_latent_y_CCA, X_latent_var_accum, y_latent_var_accum, corr_latent_CCA_accum = flip_latent_var_CCA(
        X_latent_var_list = X_latent_var_list, 
        y_latent_var_list = y_latent_var_list
    )
    
    flipped_y_weight = flipped_weights_accum[i]
    y_weights_data = np.array(flipped_y_weight)

    # Transform the depression scores in ML data using the weights from CCA
    transformed_y_ML_depr = y_data_deconf[:, :16] * y_weights_data

    # Print the best hyperparameter 
    print(f'Information for entire CCA data {i+1}:\n')
    print(f'Best estimator: {best_model}')

    # Print CCA canonical correlations for each dimension
    for comp, cor in enumerate(CCA_canonical_corrs):
        print(f'CCA canonical correlations for dimension {comp+1}: {cor}') 
    
    # Print explained variance for each dimension
    print(f'Explained variance for sleep (X) in fold {i+1}: {exp_var_X1:.4f}')
    print(f'Explained variance for depression (Y) in fold {i+1}: {exp_var_Y1:.4f}')

    # Print the final results
    print("CCA Canonical Loadings for X:")
    for comp, loadings in enumerate(X_loadings_CCA):
        print(f"Dimension {comp + 1}:")
        print(loadings)
    print("CCA Canonical Loadings for Y:")
    for comp, loadings in enumerate(y_loadings_CCA):
        print(f"Dimension {comp + 1}:")
        print(loadings)

    # Print correlation coefficients
    print(f'\nCoefficients for loadings between 1st and {i+1}_fold:')
    for comp, corr in enumerate([corr]):
        print(corr)

    # Print the flipped loadings
    print("\nFlipped CCA Canonical Loadings for X:")
    for comp, flipped_loadings in enumerate([flipped_X]):
        print(f"Dimension {comp + 1}:")
        print(flipped_loadings)
    print("Flipped CCA Canonical Loadings for Y:")
    for comp, flipped_loadings in enumerate([flipped_y]):
        print(f"Dimension {comp + 1}:")
        print(flipped_loadings)
    
    print(f'Weights for depression: {y_weights}\n')
    print(f'Flippet weight for depression: {flipped_y_weight}\n')
    
    # View the first 5 rows
    print("First 5 rows of transformed_y_ML:")
    print(transformed_y_ML_depr[:5])

    print(f"Latent var for depression in CCA fold {i+1}: {y_latent_var[:5]}\n")
    print(f"Flipped latent var for depression in CCA fold {i+1}: {y_latent_var[:5]}\n")

    # Get the original column names
    original_depression_columns = y_set

    # Save the ML and CCA data seperately
    save_split_datasets(
        df=df,
        ML_idx=ML_idx,
        CCA_idx=CCA_idx,
        transformed_y_ML_depr=transformed_y_ML_depr, 
        original_column_names=original_depression_columns, 
        i=i
        )

    # Save the latent variables and loadings for each view
    all_CCA_latent_df = save_info_for_each_fold(
        i=i, 
        CCA_idx=CCA_idx, 
        X_latent_var=flipped_latent_X_CCA,
        y_latent_var=flipped_latent_y_CCA,
        all_CCA_latent_df=all_CCA_latent_df
        )

    all_loadings_df = save_loadings_for_each_fold(
        i=i, 
        X_loadings=X_loadings_CCA, 
        y_loadings=y_loadings_CCA, 
        all_loadings_df=all_loadings_df
        )
    
    # Calculate the redundancy index and explained variane by depressive symptoms
    # Redundancy index: https://brainder.org/2019/12/27/redundancy-in-canonical-correlation-analysis/
    X_loading_sleep = np.array(X_loadings_CCA)
    y_loading_depression = np.array(y_loadings_CCA)

    VE_sleep = np.mean(X_loading_sleep**2, axis=1)
    VE_depression = np.mean(y_loading_depression**2, axis=1)

    redundancy = VE_sleep * CCA_canonical_corrs**2

    explained_variance_sleep_list.append(VE_sleep)
    explained_variance_depression_list.append(VE_depression)
    redundancy_index_list.append(redundancy)


    # Print out explained variance
    print(f'Explained variance for sleep in fold {i+1}: {VE_sleep}')
    print(f'Explained variance for depressive symptoms in fold {i+1}: {VE_depression}')
    print(f'Redundancy index for depressive symptoms in fold {i+1}: {redundancy}')
    
    # Save the updated pipeline 
    joblib.dump(best_model, f'path_to_project_root/rcca_{i+1}.joblib')

    # ============================================================================================ #
    # Transform the trained CCA model on ML fold (test fold) for model evaluation
    best_model.transform([X_ML, y_ML])

    # Calculate canonical correlations for the ML data for each fold
    CCA_canonical_corrs_ML = canonical_correlations_CCA(best_model, X_ML, y_ML)
    CCA_correlations_ML.append(CCA_canonical_corrs_ML)

    # Get loadings for each dimension for the ML data
    X_loadings_ML, y_loadings_ML = collect_canonical_loadings_CCA(best_model, n_latent_dimension, X_ML, y_ML)

    loadingsX_ML_list.append(X_loadings_ML)
    loadingsY_ML_list.append(y_loadings_ML)

    # Calculate the latent variables for sleep and depression
    X_latent_var_ML, y_latent_var_ML = latent_var(best_model, X_ML, y_ML)
    X_latent_var_ML_list.append(X_latent_var_ML)
    y_latent_var_ML_list.append(y_latent_var_ML)

    # Flit loadings in ML data for later plotting
    corr_ML, flipped_X_ML, flipped_y_ML, X_loadings_ML_accum, y_loadings_ML_accum = flip_loadings_ML(
        X_loadings_ML_list = loadingsX_ML_list, 
        y_loadings_ML_list = loadingsY_ML_list)
    
    # Flip the latent variates for ML fold to ensure each fold is in the same direction with the 1st one
    corr_latent_var_ML, flipped_latent_X_ML, flipped_latent_y_ML, X_latent_var_ML_accum, y_latent_var_ML_accum, corr_latent_ML_accum = flip_latent_var_ML(
        X_latent_var_ML_list = X_latent_var_ML_list, 
        y_latent_var_ML_list = y_latent_var_ML_list
    )

    print(f"Latent var for depression in ML fold {i+1}: {y_latent_var_ML[:5]}\n")
    print(f"Flipped latent var for depression in ML fold {i+1}: {flipped_latent_y_ML[:5]}\n")

    # Print CCA canonical correlations for each dimension
    for comp, cor in enumerate(CCA_canonical_corrs_ML):
        print(f'CCA canonical correlations for dimension {comp+1} in ML fold: {cor}')

    # Save the latent variables and loadings for each view in ML data 
    all_CCA_latent_ML_df = save_info_for_each_fold(
        i=i, 
        CCA_idx=ML_idx, 
        X_latent_var=flipped_latent_X_ML,
        y_latent_var=flipped_latent_y_ML,
        all_CCA_latent_df=all_CCA_latent_ML_df
        )
    
    all_loadings_ML_df = save_loadings_for_each_fold(
        i=i, 
        X_loadings=X_loadings_ML, 
        y_loadings=y_loadings_ML, 
        all_loadings_df=all_loadings_ML_df
        )
    # ============================================================================================ #
    # Train rCCA model on the shuffled y in CCA data
    perm_corrs = permutation_test_rcca (best_model, X_CCA, y_CCA, X_ML, y_ML, n_perm = 1000)

    # Empirical p value to estimate significance
    p_value = (1 + np.sum(perm_corrs >= CCA_canonical_corrs_ML)) / (1 + len(perm_corrs))
    permuted_pvals.append(p_value)

    # Print results
    print(f"\nObserved canonical correlation: {perm_corrs[:5]}")
    print(f"Permutation p-value: {p_value}")

mean_GP1 = np.mean(CCA_correlations_ML)
std = np.std(CCA_correlations_ML)
print(f'Mean [std] for canonical correlation coefficient across test sets in GP1: {mean_GP1:.3f} [{std:.3f}]')

# Calculate the FDR-corrected p for permuted p
rejected, pvals_fdr = fdrcorrection(permuted_pvals, alpha=0.05)

# Print the FDR-corrected p and r 
print(f"\nStatistical significance for CCA after FDR correction: {pvals_fdr}")

# Calculate the averaged weight (for OOSV in clinical and longitudinal data)
averaged_weight = np.mean(flipped_weights_accum, axis=0)

# Stack the GP_corss and confounds
entire_X_data = np.hstack((X_data, join_confounds))
entire_y_data = np.hstack((y_data, join_confounds))

# Apply preprocessing and deconfounding to the y data
y_preprocessed = best_model.named_steps['preprocessor'].transform([entire_X_data, entire_y_data])
y_deconfounded = best_model.named_steps['deconfound'].transform([y_preprocessed[0], y_preprocessed[1]])

# Extract the preprocessed depression data for y data
y_deconf = y_deconfounded[1]

# Calculate the SRD for the entire GP_cross data
transformed_y_depr = y_deconf[:, :16] * averaged_weight

# Add transformed depression columns to ML data
for j, col_name in enumerate(original_depression_columns):
    df[f'Trans_{col_name}'] = transformed_y_depr[:, j]

# Save the accumulated dataframe to csv file
all_CCA_latent_df.to_csv('working_path/CCA_latent_vars.csv', index=False)
all_loadings_ML_df.to_csv('working_path/CCA_loadings.csv', index=False)
all_CCA_latent_ML_df.to_csv('working_path/CCA_latent_vars_ML.csv', index=False)
all_loadings_df.to_csv('working_path/CCA_loadings.csv', index=False)
weight_y_df.to_csv('working_path/weight_y.csv', index=False)